# LinkedIn Skills

In this chapter, skills are standardised LinkedIn skill labels drawn from members' profiles. Expert taxonomists standardize skill inputs into approximately 38,000 skills, categorized into 249 skill groups. Skill group descriptions appear in the Appendix of [World Bank Group-LinkedIn Data Insights](https://documents1.worldbank.org/curated/en/827991542143093021/pdf/World-Bank-Group-LinkedIn-Data-Insights-Jobs-Skills-and-Migration-Trends-Methodology-and-Validation-Results.pdf).

The Skills Genome is an ordered list of the skills that are most characteristic of an entity, such as a country-industry pair. LinkedIn calculates this list with term frequency-inverse document frequency (TF-IDF), which gives more weight to distinctive skills and down-ranks skills that are common across many entities.

TF-IDF is a statistical measure that evaluates how representative a word (in this case a skill) is to a selected entity by multiplying two metrics:
1. The term frequency of a skill in an entity (`TF`).
2. The logarithmic inverse entity frequency of the skill across a set of entities (`IDF`). This indicates how common or rare a word is in the entire entity set. The closer IDF is to zero, the more common a word is.

Therefore, if the skill is very common across LinkedIn entities, and appears in many job or member descriptions, the `IDF` will approach zero. If, on the other hand, the skill is unique to specific entities, the `IDF` will approach one. Further details are available in [LinkedIn's Skills Genome Description](https://engineering.linkedin.com/blog/2019/how-we-mapped-the-skills-genome-of-emerging-jobs) and the [LinkedIn-World Bank Methodology note](https://documents1.worldbank.org/curated/en/827991542143093021/pdf/World-Bank-Group-LinkedIn-Data-Insights-Jobs-Skills-and-Migration-Trends-Methodology-and-Validation-Results.pdf).
 
Pooled skills use skills added across all years in the reporting period, while flow skills use skills added during an individual year.

In [1]:
from pathlib import Path

import altair as alt
import attaviz
import pandas as pd

attaviz.enable()
alt.data_transformers.enable("vegafusion")


def find_project_root(marker="pyproject.toml"):
    current = Path.cwd()
    for parent in [current, *current.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "LinkedIn"
PROCESSED_PATH = DATA_PATH / "processed"
PROCESSED_PATH.mkdir(exist_ok=True)
SKILLS_FILE = (
    DATA_PATH
    / "LinkedIn Skills Genome and Penetration"
    / "Skills Genome and Skills Pen 2026.xlsx"
)

WEST_AFRICA = ["Ghana", "Nigeria"]
COMPARATORS = ["India", "Kenya", "South Africa"]
COUNTRIES = WEST_AFRICA + COMPARATORS
COUNTRY_COLOURS = dict(zip(COUNTRIES, attaviz.CATEGORICAL))

SKILL_ORDER = [
    "Soft Skills",
    "Tech Skills",
    "Business Skills",
    "Disruptive Tech Skills",
    "Green Skills",
]
GITHUB_PATH = PROJECT_ROOT / "data" / "GitHub"
TARGET_INDUSTRIES = [
    "Technology, Information and Media",
    "Professional Services",
]
TOP_SKILLS = 10
BENCHMARK = 1.0
SKILLS_NOTE = (
    "Source: LinkedIn Economic Graph, Skills Genome and Skills Penetration 2026."
)

In [2]:
def read_skills(sheet_name, header, names=None, countries=None):
    """Read one Skills Genome sheet and return a tidy frame."""
    data = pd.read_excel(SKILLS_FILE, sheet_name=sheet_name, header=header).dropna(
        axis="columns", how="all"
    )
    if names is not None:
        data = data.set_axis(names, axis="columns")
    else:
        data = data.loc[
            :, [c for c in data.columns if not str(c).startswith("Unnamed")]
        ]
    return (
        data.loc[lambda d: d["Country"].ne("Country")]
        .dropna(subset=data.columns.tolist())
        .assign(
            **{
                c: lambda d, c=c: d[c].astype(str).str.strip()
                for c in data.columns
                if pd.api.types.is_string_dtype(data[c])
            }
        )
        .loc[lambda d: d["Country"].isin(countries) if countries else slice(None)]
        .drop_duplicates()
        .reset_index(drop=True)
    )


def read_comparators():
    """Return the workbook's comparator list, one row per country."""
    return (
        pd.read_excel(SKILLS_FILE, sheet_name="Ref - Country Comparators", header=3)
        .dropna(axis="columns", how="all")
        .dropna(subset=["Country"])
        .set_index("Country")
    )


def skill_scale(domain):
    """Use consistent country colours across all skills charts."""
    domain = list(domain)
    return alt.Scale(domain=domain, range=[COUNTRY_COLOURS[c] for c in domain])


def publication_table(data, caption, formats=None, wrap_columns=None):
    """Format a dataframe for stable HTML rendering in Jupyter Book."""
    styler = data.style.hide(axis="index").set_caption(caption)
    if formats:
        styler = styler.format(formats, na_rep="—")
    styler = styler.set_properties(
        **{
            "font-size": "0.875rem",
            "line-height": "1.4",
            "padding": "0.45rem 0.6rem",
            "text-align": "left",
            "vertical-align": "top",
            "border-bottom": "1px solid #e5e7eb",
        }
    ).set_table_styles(
        [
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("width", "100%"),
                    ("margin-bottom", "1.5rem"),
                ],
            },
            {
                "selector": "caption",
                "props": [
                    ("caption-side", "top"),
                    ("text-align", "left"),
                    ("font-size", "1rem"),
                    ("font-weight", "600"),
                    ("padding-bottom", "0.5rem"),
                ],
            },
            {
                "selector": "th",
                "props": [
                    ("background-color", "#f3f4f6"),
                    ("font-weight", "600"),
                    ("padding", "0.45rem 0.6rem"),
                    ("text-align", "left"),
                    ("vertical-align", "bottom"),
                    ("border-bottom", "2px solid #d1d5db"),
                ],
            },
            {
                "selector": "tbody tr:nth-child(even)",
                "props": [("background-color", "#f9fafb")],
            },
        ]
    )
    if wrap_columns:
        styler = styler.set_properties(
            subset=wrap_columns,
            **{"white-space": "normal", "min-width": "12rem"},
        )
    return styler

## GitHub languages in LinkedIn skills

GitHub labels are first classified as programming, data, or markup, with both the GitHub and LinkedIn sides restricted to five countries: Ghana, Nigeria, India, Kenya, and South Africa. It searches pooled country-industry, annual-flow, and gender-disaggregated LinkedIn skill profiles across every industry and rank. Labels must match after case and whitespace normalisation, except for four explicit aliases used by LinkedIn (`C (Programming Language)`, `Python (Programming Language)`, `Cascading Style Sheets (CSS)`, and `Shell Scripting`).

### All matched GitHub and LinkedIn labels

This table shows every conservative GitHub-LinkedIn label match found across the available period. Each row identifies the LinkedIn label and lists the actual country-industry locations in which LinkedIn reports it.

- GitHub Language: the label used in GitHub's languages data.
- LinkedIn Skill: the exact LinkedIn skill label to which it was matched.
- GitHub Type: GitHub's classification of the label as programming, markup, or data.
- Match Method: whether the labels match directly after normalisation or through one of the four documented aliases.
- LinkedIn Datasets: the Skills Genome sources in which the match appears: pooled country-industry, annual flow, and/or pooled by gender.
- Countries and industries: the selected countries and specific LinkedIn industries in which the skill appears.

### Twenty most active GitHub labels and their LinkedIn visibility

This table shows the 20 GitHub labels with the most pusher activity in the five countries during 2025 Q4, including labels with no LinkedIn match.

- GitHub Language / GitHub Type: the GitHub label and its GitHub classification.
- LinkedIn Skill: the matched LinkedIn label, or a dash when it is not reported.
- LinkedIn Countries / LinkedIn Industries: the number of selected countries and distinct industries in which the skill appears across the three Skills Genome datasets.
- Countries and industries: the reported locations, or `Not reported in selected LinkedIn data` when there is no match.

In [3]:
GITHUB_LANGUAGE_TYPES = ["programming", "data", "markup"]
GITHUB_COUNTRY_CODES = ["GH", "NG", "IN", "KE", "ZA"]
LANGUAGE_ALIASES = {
    "C": "C (Programming Language)",
    "Python": "Python (Programming Language)",
    "CSS": "Cascading Style Sheets (CSS)",
    "Shell": "Shell Scripting",
}

github_language_activity = pd.read_csv(GITHUB_PATH / "languages.csv").loc[
    lambda d: (
        d["language_type"].isin(GITHUB_LANGUAGE_TYPES)
        & d["iso2_code"].isin(GITHUB_COUNTRY_CODES)
    )
]
latest_github_period = (
    github_language_activity[["year", "quarter"]]
    .drop_duplicates()
    .sort_values(["year", "quarter"])
    .iloc[-1]
)
latest_github_activity = (
    github_language_activity.loc[
        lambda d: (
            d["year"].eq(latest_github_period["year"])
            & d["quarter"].eq(latest_github_period["quarter"])
        )
    ]
    .groupby(["language", "language_type"], as_index=False)["num_pushers"]
    .sum()
    .rename(columns={"num_pushers": "Latest Pushers"})
)
github_language_inventory = (
    github_language_activity.filter(["language", "language_type"])
    .drop_duplicates()
    .merge(latest_github_activity, on=["language", "language_type"], how="left")
    .rename(columns={"language": "GitHub Language", "language_type": "GitHub Type"})
    .fillna({"Latest Pushers": 0})
    .assign(
        **{
            "LinkedIn Skill": lambda d: d["GitHub Language"].replace(LANGUAGE_ALIASES),
            "Match Method": lambda d: d["GitHub Language"].map(
                lambda value: (
                    "explicit alias"
                    if value in LANGUAGE_ALIASES
                    else "normalised exact"
                )
            ),
            "Latest Pusher Share": lambda d: (
                d["Latest Pushers"] / d["Latest Pushers"].sum()
            ),
        }
    )
)

pooled_skill_genome = read_skills(
    "2A - SGP Ctry Ind",
    header=3,
    names=["Country", "Industry", "Skill", "Skill Rank"],
    countries=COUNTRIES,
).assign(**{"Skill Rank": lambda d: pd.to_numeric(d["Skill Rank"])})
flow_skill_genome = read_skills(
    "2B - SGF Ctry Ind Yr",
    header=3,
    names=["Country", "Year", "Industry", "Skill", "Skill Rank"],
    countries=COUNTRIES,
).assign(
    Year=lambda d: pd.to_numeric(d["Year"]).astype(int),
    **{"Skill Rank": lambda d: pd.to_numeric(d["Skill Rank"])},
)
gender_skill_genome = read_skills(
    "2C - SGP Ctry Ind Gen Yr", header=5, countries=COUNTRIES
).assign(**{"Skill Rank": lambda d: pd.to_numeric(d["Skill Rank"])})


def match_github_languages(linkedin_skills, dataset):
    """Return conservative language matches with their LinkedIn locations."""
    return (
        github_language_inventory.merge(
            linkedin_skills,
            left_on=github_language_inventory["LinkedIn Skill"].str.casefold(),
            right_on=linkedin_skills["Skill"].str.casefold(),
            how="inner",
        )
        .drop(columns=["key_0"])
        .assign(**{"LinkedIn Dataset": dataset})
        .sort_values(["GitHub Language", "Country", "Industry", "Skill Rank"])
    )


pooled_language_matches = match_github_languages(
    pooled_skill_genome, "Pooled country-industry"
)
flow_language_matches = match_github_languages(flow_skill_genome, "Annual flow")
gender_language_matches = match_github_languages(
    gender_skill_genome, "Pooled by gender"
)
language_matches = pd.concat(
    [pooled_language_matches, flow_language_matches, gender_language_matches],
    ignore_index=True,
)
language_match_summary = (
    language_matches.groupby("GitHub Language", as_index=False)
    .agg(
        **{
            "LinkedIn Countries": ("Country", "nunique"),
            "LinkedIn Industries": ("Industry", "nunique"),
            "Best LinkedIn Rank": ("Skill Rank", "min"),
        }
    )
    .merge(
        flow_language_matches.groupby("GitHub Language")["Year"]
        .max()
        .rename("Latest LinkedIn Year"),
        on="GitHub Language",
        how="left",
    )
)
language_coverage = (
    github_language_inventory.merge(
        language_match_summary, on="GitHub Language", how="left"
    )
    .fillna(
        {
            "LinkedIn Countries": 0,
            "LinkedIn Industries": 0,
        }
    )
    .assign(
        **{
            "LinkedIn Countries": lambda d: d["LinkedIn Countries"].astype(int),
            "LinkedIn Industries": lambda d: d["LinkedIn Industries"].astype(int),
            "Appears in LinkedIn": lambda d: d["LinkedIn Countries"].gt(0),
        }
    )
)
latest_language_coverage = language_coverage.loc[
    language_coverage["Latest Pushers"].gt(0)
].copy()
language_coverage_summary = (
    latest_language_coverage.assign(
        **{
            "Matched Latest Pushers": lambda d: d["Latest Pushers"].where(
                d["Appears in LinkedIn"], 0
            )
        }
    )
    .groupby("GitHub Type", as_index=False)
    .agg(
        **{
            "GitHub Labels": ("GitHub Language", "nunique"),
            "Matched Labels": ("Appears in LinkedIn", "sum"),
            "Latest Pushers": ("Latest Pushers", "sum"),
            "Matched Latest Pushers": ("Matched Latest Pushers", "sum"),
        }
    )
    .assign(
        **{
            "Label Coverage": lambda d: d["Matched Labels"] / d["GitHub Labels"],
            "Activity Coverage": lambda d: (
                d["Matched Latest Pushers"] / d["Latest Pushers"]
            ),
        }
    )
)
overall_language_coverage = pd.DataFrame(
    {
        "GitHub Type": ["All included types"],
        "GitHub Labels": [latest_language_coverage["GitHub Language"].nunique()],
        "Matched Labels": [latest_language_coverage["Appears in LinkedIn"].sum()],
        "Latest Pushers": [latest_language_coverage["Latest Pushers"].sum()],
        "Matched Latest Pushers": [
            latest_language_coverage.loc[
                latest_language_coverage["Appears in LinkedIn"], "Latest Pushers"
            ].sum()
        ],
    }
).assign(
    **{
        "Label Coverage": lambda d: d["Matched Labels"] / d["GitHub Labels"],
        "Activity Coverage": lambda d: (
            d["Matched Latest Pushers"] / d["Latest Pushers"]
        ),
    }
)
language_coverage_summary = pd.concat(
    [language_coverage_summary, overall_language_coverage], ignore_index=True
)
matched_label_rows = []
for keys, group in language_matches.groupby(
    ["GitHub Language", "LinkedIn Skill", "GitHub Type", "Match Method"]
):
    github_language, linkedin_skill, github_type, match_method = keys
    country_locations = []
    for country in COUNTRIES:
        industries = sorted(
            group.loc[group["Country"].eq(country), "Industry"].unique()
        )
        if industries:
            country_locations.append(f"{country}: {', '.join(industries)}")
    datasets = [
        dataset
        for dataset in ["Pooled country-industry", "Annual flow", "Pooled by gender"]
        if dataset in set(group["LinkedIn Dataset"])
    ]
    matched_label_rows.append(
        {
            "GitHub Language": github_language,
            "LinkedIn Skill": linkedin_skill,
            "GitHub Type": github_type,
            "Match Method": match_method,
            "LinkedIn Datasets": ", ".join(datasets),
            "Countries and industries": "; ".join(country_locations),
        }
    )
matched_label_details = (
    pd.DataFrame(matched_label_rows)
    .sort_values("GitHub Language")
    .reset_index(drop=True)
)
top_github_visibility = (
    language_coverage.nlargest(20, "Latest Pushers")
    .merge(
        matched_label_details[["GitHub Language", "Countries and industries"]],
        on="GitHub Language",
        how="left",
    )
    .assign(
        **{
            "LinkedIn Skill": lambda d: d["LinkedIn Skill"].where(
                d["Appears in LinkedIn"]
            ),
            "Countries and industries": lambda d: d["Countries and industries"].fillna(
                "Not reported in selected LinkedIn data"
            ),
        }
    )
    .filter(
        [
            "GitHub Language",
            "GitHub Type",
            "LinkedIn Skill",
            "LinkedIn Countries",
            "LinkedIn Industries",
            "Countries and industries",
        ]
    )
)

language_matches.to_csv(
    PROCESSED_PATH / "github_linkedin_language_locations.csv", index=False
)
language_coverage.to_csv(
    PROCESSED_PATH / "github_linkedin_language_coverage.csv", index=False
)
language_coverage_summary.to_csv(
    PROCESSED_PATH / "github_linkedin_language_coverage_summary.csv", index=False
)
matched_label_details.to_csv(
    PROCESSED_PATH / "github_linkedin_matched_label_details.csv", index=False
)
top_github_visibility.to_csv(
    PROCESSED_PATH / "github_linkedin_top20_visibility.csv", index=False
)
display(
    publication_table(
        matched_label_details,
        caption="All matched GitHub and LinkedIn labels",
        wrap_columns=["LinkedIn Datasets", "Countries and industries"],
    )
)
display(
    publication_table(
        top_github_visibility,
        caption="Twenty GitHub labels with the most pushers in 2025 Q4 and their LinkedIn visibility",
        formats={
            "LinkedIn Countries": "{:.0f}",
            "LinkedIn Industries": "{:.0f}",
        },
        wrap_columns=["Countries and industries"],
    )
)


GitHub Language,LinkedIn Skill,GitHub Type,Match Method,LinkedIn Datasets,Countries and industries
C,C (Programming Language),programming,explicit alias,"Pooled country-industry, Annual flow, Pooled by gender","India: Consumer Services, Education, Entertainment Providers, Government Administration, Holding Companies, Manufacturing, Professional Services, Technology, Information and Media, Wholesale"
C#,C#,programming,normalised exact,"Pooled country-industry, Pooled by gender","South Africa: Professional Services, Technology, Information and Media"
C++,C++,programming,normalised exact,"Pooled country-industry, Annual flow, Pooled by gender","India: Consumer Services, Education, Entertainment Providers, Government Administration, Technology, Information and Media"
CSS,Cascading Style Sheets (CSS),markup,explicit alias,"Pooled country-industry, Annual flow, Pooled by gender","Ghana: Professional Services, Technology, Information and Media; Nigeria: Professional Services, Technology, Information and Media; India: Consumer Services, Education, Government Administration, Professional Services, Technology, Information and Media; Kenya: Technology, Information and Media; South Africa: Technology, Information and Media"
HTML,HTML,markup,normalised exact,"Pooled country-industry, Annual flow","Ghana: Professional Services, Technology, Information and Media; India: Consumer Services, Education, Government Administration, Professional Services, Technology, Information and Media; Kenya: Professional Services"
Java,Java,programming,normalised exact,"Pooled country-industry, Annual flow, Pooled by gender","India: Consumer Services, Education, Government Administration, Holding Companies, Professional Services, Technology, Information and Media"
JavaScript,JavaScript,programming,normalised exact,"Pooled country-industry, Annual flow, Pooled by gender","Ghana: Professional Services, Technology, Information and Media; Nigeria: Professional Services, Technology, Information and Media; India: Consumer Services, Education, Government Administration, Professional Services, Technology, Information and Media; Kenya: Professional Services, Technology, Information and Media; South Africa: Technology, Information and Media"
MATLAB,MATLAB,programming,normalised exact,"Pooled country-industry, Annual flow, Pooled by gender","Ghana: Utilities; India: Consumer Services, Education"
PHP,PHP,programming,normalised exact,Pooled by gender,Kenya: Professional Services
Python,Python (Programming Language),programming,explicit alias,"Pooled country-industry, Annual flow, Pooled by gender","Ghana: Education, Professional Services, Technology, Information and Media; Nigeria: Education, Holding Companies; India: Administrative and Support Services, Consumer Services, Education, Entertainment Providers, Government Administration, Holding Companies, Professional Services, Technology, Information and Media, Wholesale; Kenya: Education, Professional Services, Technology, Information and Media"


GitHub Language,GitHub Type,LinkedIn Skill,LinkedIn Countries,LinkedIn Industries,Countries and industries
HTML,markup,HTML,3,5,"Ghana: Professional Services, Technology, Information and Media; India: Consumer Services, Education, Government Administration, Professional Services, Technology, Information and Media; Kenya: Professional Services"
JavaScript,programming,JavaScript,5,5,"Ghana: Professional Services, Technology, Information and Media; Nigeria: Professional Services, Technology, Information and Media; India: Consumer Services, Education, Government Administration, Professional Services, Technology, Information and Media; Kenya: Professional Services, Technology, Information and Media; South Africa: Technology, Information and Media"
CSS,markup,Cascading Style Sheets (CSS),5,5,"Ghana: Professional Services, Technology, Information and Media; Nigeria: Professional Services, Technology, Information and Media; India: Consumer Services, Education, Government Administration, Professional Services, Technology, Information and Media; Kenya: Technology, Information and Media; South Africa: Technology, Information and Media"
Python,programming,Python (Programming Language),4,9,"Ghana: Education, Professional Services, Technology, Information and Media; Nigeria: Education, Holding Companies; India: Administrative and Support Services, Consumer Services, Education, Entertainment Providers, Government Administration, Holding Companies, Professional Services, Technology, Information and Media, Wholesale; Kenya: Education, Professional Services, Technology, Information and Media"
TypeScript,programming,TypeScript,2,1,"Ghana: Technology, Information and Media; Nigeria: Technology, Information and Media"
Java,programming,Java,1,6,"India: Consumer Services, Education, Government Administration, Holding Companies, Professional Services, Technology, Information and Media"
Shell,programming,Shell Scripting,1,2,"India: Professional Services, Technology, Information and Media"
Jupyter Notebook,markup,—,0,0,Not reported in selected LinkedIn data
Dockerfile,programming,—,0,0,Not reported in selected LinkedIn data
C++,programming,C++,1,5,"India: Consumer Services, Education, Entertainment Providers, Government Administration, Technology, Information and Media"


In [4]:
RELATED_SKILL_TAXONOMY = {
    "Programming languages": [
        "C (Programming Language)",
        "Python (Programming Language)",
    ],
    "Software engineering": [
        "Agile Web Development",
        "Back-End Web Development",
        "Software Development",
        "Software Development Life Cycle (SDLC)",
        "Web Development",
    ],
    "Data analysis": [
        "Business Intelligence",
        "Data Analysis",
        "Data Analytics",
        "Statistical Data Analysis",
    ],
    "Data science and AI": [
        "Artificial Intelligence (AI)",
        "Data Science",
        "Machine Learning",
    ],
    "Databases": [
        "Database Management System (DBMS)",
        "Microsoft SQL Server",
        "MySQL",
        "PL/SQL",
        "SQL",
    ],
    "Cloud and infrastructure": ["Cloud Computing"],
}
related_skill_lookup = pd.DataFrame(
    [
        {"Skill Category": category, "Skill": skill}
        for category, skills in RELATED_SKILL_TAXONOMY.items()
        for skill in skills
    ]
)
related_software_data_skills = (
    pd.concat(
        [
            pooled_skill_genome.assign(Period="Pooled 2017–2025", Year=pd.NA),
            flow_skill_genome.assign(Period="Annual flow"),
        ],
        ignore_index=True,
    )
    .merge(related_skill_lookup, on="Skill", how="inner")
    .sort_values(["Skill Category", "Skill", "Country", "Industry", "Period", "Year"])
)
related_software_data_skills.to_csv(
    PROCESSED_PATH / "related_software_data_skill_locations.csv", index=False
)
display(related_software_data_skills)


,Country,Industry,Skill,Skill Rank,Period,Year,Skill Category
279,Ghana,Professional Services,Cloud Computing,21,Annual flow,2025,Cloud and infrastructure
284,Ghana,"Technology, Information and Media",Cloud Computing,19,Annual flow,2025,Cloud and infrastructure
240,Kenya,"Technology, Information and Media",Cloud Computing,9,Annual flow,2019,Cloud and infrastructure
334,Kenya,"Technology, Information and Media",Cloud Computing,23,Annual flow,2025,Cloud and infrastructure
53,Kenya,"Technology, Information and Media",Cloud Computing,22,Pooled 2017–2025,<NA>,Cloud and infrastructure
...,...,...,...,...,...,...,...
263,Nigeria,"Technology, Information and Media",Web Development,1,Annual flow,2021,Software engineering
264,Nigeria,"Technology, Information and Media",Web Development,8,Annual flow,2022,Software engineering
265,Nigeria,"Technology, Information and Media",Web Development,5,Annual flow,2024,Software engineering
337,Nigeria,"Technology, Information and Media",Web Development,8,Annual flow,2025,Software engineering


## Data Annotation Skill

The table below covers every industry and rank across the pooled country-industry, annual-flow, and gender-disaggregated Skills Genome data for Ghana, Nigeria, India, Kenya, and South Africa. The status table distinguishes genuine non-matches from countries unavailable in a source sheet.

In [5]:
annotation_datasets = {
    "Pooled country-industry": pooled_skill_genome.assign(Year=pd.NA, Gender=pd.NA),
    "Annual flow": flow_skill_genome.assign(Gender=pd.NA),
    "Pooled by gender": gender_skill_genome.assign(Year=pd.NA),
}
annotation_combined = pd.concat(
    [data.assign(Dataset=name) for name, data in annotation_datasets.items()],
    ignore_index=True,
)
data_annotation_locations = (
    annotation_combined.loc[lambda d: d["Skill"].str.casefold().eq("data annotation")]
    .sort_values(["Country", "Industry", "Dataset", "Year", "Gender"])
    .filter(["Country", "Industry", "Dataset", "Year", "Gender", "Skill", "Skill Rank"])
)
annotation_status = pd.MultiIndex.from_product(
    [COUNTRIES, annotation_datasets], names=["Country", "Dataset"]
).to_frame(index=False)
available_country_datasets = (
    annotation_combined[["Country", "Dataset"]].drop_duplicates().assign(Available=True)
)
found_country_datasets = (
    data_annotation_locations[["Country", "Dataset"]]
    .drop_duplicates()
    .assign(Found=True)
)
annotation_status = (
    annotation_status.merge(
        available_country_datasets, on=["Country", "Dataset"], how="left"
    )
    .merge(found_country_datasets, on=["Country", "Dataset"], how="left")
    .fillna({"Available": False, "Found": False})
    .assign(
        Status=lambda d: d.apply(
            lambda row: (
                "Found"
                if row["Found"]
                else "Not found in reported top-30 skills"
                if row["Available"]
                else "Data unavailable"
            ),
            axis=1,
        )
    )
    .filter(["Country", "Dataset", "Status"])
)
ANNOTATION_ADJACENT_SKILLS = [
    "Annotation",
    "Data Entry",
    "Online Data Entry",
    "Transcription",
    # "Machine Learning",
]
annotation_adjacent_locations = (
    annotation_combined.loc[lambda d: d["Skill"].isin(ANNOTATION_ADJACENT_SKILLS)]
    .sort_values(["Skill", "Country", "Industry", "Dataset", "Year", "Gender"])
    .filter(["Country", "Industry", "Dataset", "Year", "Gender", "Skill", "Skill Rank"])
)
annotation_adjacent_summary = (
    annotation_adjacent_locations.groupby(["Skill", "Dataset"], as_index=False)
    .agg(
        **{
            "Countries": ("Country", "nunique"),
            "Industries": ("Industry", "nunique"),
            "Best Rank": ("Skill Rank", "min"),
        }
    )
    .sort_values(["Skill", "Dataset"])
)
data_annotation_locations.to_csv(
    PROCESSED_PATH / "data_annotation_locations.csv", index=False
)
annotation_status.to_csv(PROCESSED_PATH / "data_annotation_status.csv", index=False)
annotation_adjacent_locations.to_csv(
    PROCESSED_PATH / "data_annotation_adjacent_skill_locations.csv", index=False
)
annotation_adjacent_summary.to_csv(
    PROCESSED_PATH / "data_annotation_adjacent_skill_summary.csv", index=False
)
annotation_status_display = (
    annotation_status.pivot(index="Country", columns="Dataset", values="Status")
    .reset_index()
    .rename_axis(columns=None)
)
display(
    publication_table(
        annotation_status_display,
        caption="Data Annotation reporting status by country and LinkedIn dataset",
        wrap_columns=[
            "Annual flow",
            "Pooled by gender",
            "Pooled country-industry",
        ],
    )
)
display(
    publication_table(
        data_annotation_locations,
        caption="Reported Data Annotation locations",
        formats={"Year": "{:.0f}", "Skill Rank": "{:.0f}"},
        wrap_columns=["Industry", "Dataset"],
    )
)
display(
    publication_table(
        annotation_adjacent_summary,
        caption="Annotation-adjacent skills by LinkedIn dataset",
        formats={
            "Countries": "{:.0f}",
            "Industries": "{:.0f}",
            "Best Rank": "{:.0f}",
        },
        wrap_columns=["Dataset"],
    )
)


Country,Annual flow,Pooled by gender,Pooled country-industry
Ghana,Not found in reported top-30 skills,Found,Not found in reported top-30 skills
India,Not found in reported top-30 skills,Not found in reported top-30 skills,Not found in reported top-30 skills
Kenya,Found,Found,Found
Nigeria,Not found in reported top-30 skills,Data unavailable,Not found in reported top-30 skills
South Africa,Not found in reported top-30 skills,Not found in reported top-30 skills,Not found in reported top-30 skills


Country,Industry,Dataset,Year,Gender,Skill,Skill Rank
Ghana,Professional Services,Pooled by gender,—,Female,Data Annotation,28
Kenya,Professional Services,Annual flow,2024,—,Data Annotation,4
Kenya,Professional Services,Annual flow,2025,—,Data Annotation,4
Kenya,Professional Services,Pooled by gender,—,Female,Data Annotation,8
Kenya,Professional Services,Pooled by gender,—,Male,Data Annotation,2
Kenya,Professional Services,Pooled country-industry,—,—,Data Annotation,5
Kenya,"Technology, Information and Media",Annual flow,2024,—,Data Annotation,7
Kenya,"Technology, Information and Media",Annual flow,2025,—,Data Annotation,3
Kenya,"Technology, Information and Media",Pooled by gender,—,Female,Data Annotation,9
Kenya,"Technology, Information and Media",Pooled by gender,—,Male,Data Annotation,2


Skill,Dataset,Countries,Industries,Best Rank
Annotation,Pooled by gender,1,1,7
Data Entry,Annual flow,3,17,1
Data Entry,Pooled by gender,2,9,11
Data Entry,Pooled country-industry,3,9,3
Online Data Entry,Annual flow,2,3,6
Online Data Entry,Pooled by gender,2,5,6
Online Data Entry,Pooled country-industry,2,3,16
Transcription,Annual flow,1,2,6
Transcription,Pooled by gender,1,3,10
Transcription,Pooled country-industry,1,3,17


## Skills penetration

Skills penetration measures how strongly broad skill groups appear in an entity's characteristic skill profile. At a high level, it is the share of an entity's top 50 skills that belongs to a given skill group. For example, if 5 of 50 skills for Industry X in Country Y fall into the Artificial Intelligence skill group, Artificial Intelligence has a 10% penetration for Industry X and Country Y. To produce comparable country-industry estimates, LinkedIn identifies representative skills for each industry-occupation pair, assigns them to skill groups, and then aggregates the results.

The charts use relative skills penetration, where national penetration divided by a comparable global benchmark that accounts for the industry's occupational mix. A value of 1.0 matches the global benchmark, a value above 1.0 indicates higher penetration, and a value below 1.0 indicates lower penetration.

### Pooled by country (2017–2025)

In [6]:
comparators = read_comparators()
skill_penetration = read_skills("3A - SPP Ctry", header=5, countries=COUNTRIES).assign(
    **{c: lambda d, c=c: pd.to_numeric(d[c]) for c in ["Average", "Global", "Relative"]}
)
skill_penetration.to_csv(PROCESSED_PATH / "skill_penetration_country.csv", index=False)


def plot_country_penetration(data, width=520, height=250):
    """Plot relative skill-group penetration for the selected countries."""
    countries = [c for c in COUNTRIES if c in set(data["Country"])]
    highlight = alt.selection_point(fields=["Country"], bind="legend", name="pick")
    top = data["Relative"].max() * 1.08

    bars = (
        alt.Chart(data)
        .mark_bar()
        .encode(
            y=alt.Y("Skill:N", title=None, sort=SKILL_ORDER),
            yOffset=alt.YOffset("Country:N", sort=countries),
            x=alt.X(
                "mean(Relative):Q",
                title="Relative penetration",
                scale=alt.Scale(domain=[0, top]),
            ),
            color=alt.Color("Country:N", scale=skill_scale(countries), title=None),
            opacity=alt.when(highlight).then(alt.value(1)).otherwise(alt.value(0.25)),
            tooltip=[
                "Country:N",
                "Skill:N",
                alt.Tooltip("mean(Relative):Q", format=".2f", title="Relative"),
            ],
        )
        .add_params(highlight)
    )
    parity = (
        alt.Chart(data)
        .transform_aggregate(_rows="count()")
        .mark_rule(color=attaviz.REFERENCE, strokeDash=[4, 4])
        .encode(x=alt.datum(BENCHMARK))
    )
    chart = alt.layer(bars, parity).properties(
        width=width,
        height=height,
        title="Relative skill-group penetration against the global benchmark",
    )
    return attaviz.add_caption(
        chart,
        ["Dashed line is parity with the global benchmark (1.0).", SKILLS_NOTE],
        align="left",
    )


plot_country_penetration(skill_penetration)

alt.VConcatChart(...)

### Pooled by country and industry (2017–2025)

This view applies the same relative penetration measure within industries. Select an industry from the menu to compare the selected countries.

In [7]:
industry_penetration = read_skills(
    "3B - SPP Ctry Ind", header=5, countries=COUNTRIES
).assign(
    **{c: lambda d, c=c: pd.to_numeric(d[c]) for c in ["Average", "Global", "Relative"]}
)
industry_penetration.to_csv(
    PROCESSED_PATH / "skill_penetration_industry.csv", index=False
)


def plot_industry_penetration(data, width=520, height=250):
    """Plot relative penetration by skill group and selected industry."""
    countries = [c for c in COUNTRIES if c in set(data["Country"])]
    industries = sorted(data["Industry"].unique())
    pick = alt.selection_point(
        fields=["Industry"],
        bind=alt.binding_select(options=industries, name="Industry  "),
        value=[{"Industry": industries[0]}],
    )
    top = data["Relative"].max() * 1.08
    base = alt.Chart(data).transform_filter(pick)
    bars = base.mark_bar().encode(
        y=alt.Y("Skill:N", title=None, sort=SKILL_ORDER),
        yOffset=alt.YOffset("Country:N", sort=countries),
        x=alt.X(
            "Relative:Q", title="Relative penetration", scale=alt.Scale(domain=[0, top])
        ),
        color=alt.Color("Country:N", scale=skill_scale(countries), title=None),
        tooltip=[
            "Country:N",
            "Industry:N",
            "Skill:N",
            alt.Tooltip("Relative:Q", format=".2f"),
            alt.Tooltip("Average:Q", format=".3f", title="National"),
            alt.Tooltip("Global:Q", format=".3f", title="Global"),
        ],
    )
    parity = (
        base.transform_aggregate(_rows="count()")
        .mark_rule(color=attaviz.REFERENCE, strokeDash=[4, 4])
        .encode(x=alt.datum(BENCHMARK))
    )
    chart = (
        alt.layer(bars, parity)
        .add_params(pick)
        .properties(
            width=width,
            height=height,
            title="Relative skill-group penetration by industry",
        )
    )
    return attaviz.add_caption(
        chart,
        ["Dashed line is parity with the global benchmark (1.0).", SKILLS_NOTE],
        align="left",
    )


plot_industry_penetration(industry_penetration)

alt.VConcatChart(...)

### Relative importance in gender skill profiles

Relative importance measures how strongly each skill group characterises women's or men's profiles within an industry. It uses TF-IDF scores, so it highlights skills that are distinctive rather than simply common.

For each gender-industry pair, LinkedIn takes the top 30 characteristic skills, weights each skill by its share of the total TF-IDF score, and aggregates them into broader skill groups. A higher value means that the skill group contributes more to what makes that profile distinctive.

In [8]:
gender_skills = read_skills(
    "3B - SPP Ctry Ind Gen", header=5, countries=COUNTRIES
).assign(
    **{
        c: lambda d, c=c: pd.to_numeric(d[c])
        for c in ["Relative Importance", "Skill Group Penetration"]
    },
    Gender=lambda d: d["Gender"].str.title(),
)
gender_skills.to_csv(PROCESSED_PATH / "gender_skill_profile.csv", index=False)


def plot_gender_profile(data, width=280, height=170, panels_per_row=2):
    """Compare female and male relative importance by skill group."""
    wide = (
        data.pivot_table(
            index=["Country", "Industry", "Skill"],
            columns="Gender",
            values="Relative Importance",
            aggfunc="mean",
        )
        .reset_index()
        .rename_axis(columns=None)
    )
    paired = wide.dropna(subset=["Female", "Male"])
    industries = sorted(wide["Industry"].unique())
    countries = [c for c in COUNTRIES if c in set(wide["Country"])]
    default_industry = wide.groupby("Industry").size().idxmax()
    pick = alt.selection_point(
        fields=["Industry"],
        bind=alt.binding_select(options=industries, name="Industry  "),
        value=[{"Industry": default_industry}],
    )
    top = wide[["Female", "Male"]].max().max() * 1.08
    x_title = "Relative importance (TF-IDF weighted)"
    skills = [s for s in SKILL_ORDER if s in set(wide["Skill"])]

    def panel(country, first):
        y = alt.Y(
            "Skill:N",
            title=None,
            scale=alt.Scale(domain=skills),
            axis=alt.Axis(labels=first),
        )
        x_scale = alt.Scale(domain=[0, top])
        connector = (
            alt.Chart(paired.loc[paired["Country"].eq(country)])
            .transform_filter(pick)
            .mark_rule(color=attaviz.GREY_300, strokeWidth=2)
            .encode(y=y, x=alt.X("Female:Q", title=x_title, scale=x_scale), x2="Male:Q")
        )
        dots = (
            alt.Chart(wide.loc[wide["Country"].eq(country)])
            .transform_filter(pick)
            .transform_fold(["Female", "Male"], as_=["Gender", "Relative Importance"])
            .transform_filter("isValid(datum['Relative Importance'])")
            .mark_point(size=90, filled=True, opacity=1)
            .encode(
                y=y,
                x=alt.X("Relative Importance:Q", title=x_title, scale=x_scale),
                color=alt.Color(
                    "Gender:N",
                    scale=alt.Scale(
                        domain=["Female", "Male"],
                        range=[attaviz.GENDER["female"], attaviz.GENDER["male"]],
                    ),
                    title=None,
                ),
                tooltip=[
                    "Country:N",
                    "Industry:N",
                    "Skill:N",
                    "Gender:N",
                    alt.Tooltip("Relative Importance:Q", format=".3f"),
                ],
            )
        )
        return alt.layer(connector, dots).properties(
            width=width, height=height, title=country
        )

    rows = []
    for start in range(0, len(countries), panels_per_row):
        row_countries = countries[start : start + panels_per_row]
        rows.append(
            alt.hconcat(
                *(panel(country, i == 0) for i, country in enumerate(row_countries)),
                spacing=20,
            )
        )
    chart = (
        alt.vconcat(*rows, spacing=28)
        .add_params(pick)
        .resolve_scale(x="shared")
        .properties(title="Relative importance of skill groups by gender")
    )
    return attaviz.add_caption(
        chart,
        [
            "A skill group with one dot is reported for that gender only; a missing row is not reported. Nigeria is absent from this sheet.",
            SKILLS_NOTE,
        ],
        align="left",
    )


plot_gender_profile(gender_skills)

/var/folders/q1/wt8mfyzs73l2r5977rk_mkxm0000gn/T/ipykernel_87557/1523481037.py:107: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_gender_profile(gender_skills)
/var/folders/q1/wt8mfyzs73l2r5977rk_mkxm0000gn/T/ipykernel_87557/1523481037.py:107: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_gender_profile(gender_skills)
/Users/farhanreynaldo/Documents/world-bank/git-repo/west-africa-labor-market-analysis/.venv/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3775: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independ

alt.VConcatChart(...)

## Top skills in the target sectors

LinkedIn's latest release expands each Skills Genome list from 10 to 30 skills. To keep the tables readable and comparable with earlier releases, the following views show only ranks 1–10 for Technology, Information and Media and Professional Services. Shading identifies skills that recur across countries or years within each table.

In [9]:
TABLE_HIGHLIGHTS = list(
    dict.fromkeys(
        [
            *attaviz.CATEGORICAL,
            *(
                colour
                for palette in attaviz.REGIONS_SECONDARY.values()
                for colour in palette
            ),
        ]
    )
)
UNIQUE_FILL = attaviz.GREY_100


def readable_text_color(background):
    """Choose the attaviz text token or white for stronger contrast."""
    channels = [int(background[i : i + 2], 16) / 255 for i in (1, 3, 5)]
    linear = [
        value / 12.92 if value <= 0.04045 else ((value + 0.055) / 1.055) ** 2.4
        for value in channels
    ]
    luminance = 0.2126 * linear[0] + 0.7152 * linear[1] + 0.0722 * linear[2]
    white_contrast = 1.05 / (luminance + 0.05)
    dark_contrast = (luminance + 0.05) / 0.055
    return "#FFFFFF" if white_contrast > dark_contrast else attaviz.TEXT


def ranked_skill_table(data, columns, caption):
    """Format a rank-by-country or rank-by-year skills table."""
    grid = (
        data.sort_values([columns, "Skill Rank"])
        .pivot(index="Skill Rank", columns=columns, values="Skill")
        .rename_axis(index="Rank", columns=None)
    )
    counts = data.groupby("Skill")[columns].nunique()
    recurring = list(counts[counts > 1].index)
    colours = {
        skill: TABLE_HIGHLIGHTS[i % len(TABLE_HIGHLIGHTS)]
        for i, skill in enumerate(recurring)
    }

    def shade(skill):
        if pd.isna(skill):
            return ""
        if skill in colours:
            background = colours[skill]
            foreground = readable_text_color(background)
            return f"background-color: {background}; color: {foreground}"
        return f"background-color: {UNIQUE_FILL}; color: {attaviz.TEXT}"

    return (
        grid.style.map(shade)
        .set_caption(caption)
        .set_properties(
            **{"font-size": "11px", "padding": "4px 6px", "border": "1px solid white"}
        )
        .set_table_styles([{"selector": "th", "props": [("font-size", "11px")]}])
    )

### Pooled skills (2017–2025)

Pooled skills use skills added throughout the full reporting period. The table includes every selected country for which LinkedIn reports this sector.

In [10]:
pooled_skills = (
    read_skills(
        "2A - SGP Ctry Ind",
        header=3,
        names=["Country", "Industry", "Skill", "Skill Rank"],
        countries=COUNTRIES,
    )
    .assign(**{"Skill Rank": lambda d: pd.to_numeric(d["Skill Rank"])})
    .loc[
        lambda d: d["Industry"].isin(TARGET_INDUSTRIES) & d["Skill Rank"].le(TOP_SKILLS)
    ]
)
pooled_skills.to_csv(
    PROCESSED_PATH / "pooled_skills_target_industries_top10.csv",
    index=False,
)
for industry in TARGET_INDUSTRIES:
    industry_skills = pooled_skills.loc[pooled_skills["Industry"].eq(industry)]
    available_countries = [c for c in COUNTRIES if c in set(industry_skills["Country"])]
    display(
        ranked_skill_table(
            industry_skills.assign(
                Country=lambda d: pd.Categorical(
                    d["Country"], categories=available_countries, ordered=True
                )
            ),
            columns="Country",
            caption=f"Top 10 pooled skills in {industry}",
        )
    )

,Ghana,Nigeria,India,Kenya,South Africa
Rank,,,,,
1,React.js,Virtual Assistance,Core Java,Swahili,Afrikaans
2,Front-End Development,React.js,Amazon Web Services (AWS),Virtual Assistance,Telecommunications
3,Node.js,Virtual Administrative Support,Spring Boot,React.js,Broadcasting
4,Web Development,Search Engine Optimization (SEO),Java,Search Engine Optimization (SEO),Wireless Technologies
5,Broadcasting,Front-End Development,SQL,Python (Programming Language),Voice over IP (VoIP)
6,JavaScript,Web Content Writing,Jenkins,Agile Application Development,Software Development Life Cycle (SDLC)
7,Telecommunications,Responsive Web Design,REST APIs,Editing,IT Integration
8,Python (Programming Language),Creative Writing,C (Programming Language),Virtual Administrative Support,Television
9,Cascading Style Sheets (CSS),User Interface Design,Data Structures,Exceeding Customer Expectations,Managed Services


,Ghana,Nigeria,India,Kenya,South Africa
Rank,,,,,
1,Graphic Design,Virtual Assistance,Core Java,Swahili,Pastel Accounting
2,Ghana,Virtual Administrative Support,SQL,QuickBooks,Afrikaans
3,Python (Programming Language),Search Engine Optimization (SEO),Amazon Web Services (AWS),Virtual Assistance,Pastel Partner
4,Web Development,Legal Research,Tally ERP,Conveyancing,CaseWare Software
5,Data Entry,Graphic Design,Java,Data Annotation,Electronic Data Capture (EDC)
6,Tally ERP,User Interface Design,Jenkins,Legal Research,Civil Litigation
7,Advertising,Email Management,Spring Boot,Data Entry,Litigation
8,JavaScript,Legal Writing,Manual Testing,Search Engine Optimization (SEO),Legal Research
9,Front-End Development,User Experience Design (UED),Jira,Certified Public Accounting,Legal Writing


### Flow skills by year

Flow skills use only the skills added in each year, revealing how the sector's most characteristic new skills change over time. Each country table shows the years available in the workbook and is limited to the top 10 ranks.

In [11]:
flow_skills = (
    read_skills(
        "2B - SGF Ctry Ind Yr",
        header=3,
        names=["Country", "Year", "Industry", "Skill", "Skill Rank"],
        countries=COUNTRIES,
    )
    .assign(
        Year=lambda d: pd.to_numeric(d["Year"]).astype(int),
        **{"Skill Rank": lambda d: pd.to_numeric(d["Skill Rank"])},
    )
    .loc[
        lambda d: d["Industry"].isin(TARGET_INDUSTRIES) & d["Skill Rank"].le(TOP_SKILLS)
    ]
)
flow_skills.to_csv(
    PROCESSED_PATH / "flow_skills_target_industries_top10.csv",
    index=False,
)

for industry in TARGET_INDUSTRIES:
    industry_skills = flow_skills.loc[flow_skills["Industry"].eq(industry)]
    for country in [c for c in COUNTRIES if c in set(industry_skills["Country"])]:
        display(
            ranked_skill_table(
                industry_skills.loc[industry_skills["Country"].eq(country)],
                columns="Year",
                caption=f"Top 10 flow skills in {industry}: {country}",
            )
        )

,2017,2018,2019,2020,2021,2022,2023,2024,2025
Rank,,,,,,,,,
1,Telecommunications,Telecommunications,Telecommunications,Broadcasting,Web Development,React.js,React.js,React.js,Web Development
2,Social Media Marketing,Social Media Marketing,Graphic Design,Ghana,JavaScript,JavaScript,Cascading Style Sheets (CSS),JavaScript,Front-End Development
3,Broadcasting,Public Speaking,Web Development,Web Development,Video Editing,Cascading Style Sheets (CSS),Web Development,Python (Programming Language),React.js
4,Public Speaking,Broadcasting,Editing,Journalism,React.js,Python (Programming Language),JavaScript,Web Development,Next.js
5,Research Skills,Web Development,Video Editing,Graphic Design,Media Production,Web Development,Front-End Development,Virtual Assistance,Cascading Style Sheets (CSS)
6,Networking,Graphic Design,Broadcasting,JavaScript,Consumer Services,Graphic Design,Python (Programming Language),Front-End Development,JavaScript
7,Social Media,Team Leadership,JavaScript,Editing,Software Development,Online Advertising,Node.js,Node.js,Python (Programming Language)
8,Team Leadership,Social Media,Blogging,Telecommunications,Ghana,Node.js,HTML,Cascading Style Sheets (CSS),Virtual Assistance
9,Web Development,Networking,Journalism,Node.js,Graphic Design,HTML,Advertising,Artificial Intelligence (AI),Software Development


,2017,2018,2019,2020,2021,2022,2023,2024,2025
Rank,,,,,,,,,
1,Telecommunications,Creative Writing,Creative Writing,Web Development,Web Development,Web Content Writing,Cover Letters,Virtual Assistance,Virtual Assistance
2,Editing,Telecommunications,Web Development,Editing,Web Content Writing,Copywriting,Resume Writing,Virtual Administrative Support,Virtual Administrative Support
3,Broadcasting,Web Development,Graphic Design,Writing,Graphic Design,Virtual Assistance,Certified Professional Resume Writer,Search Engine Optimization (SEO),Email Management
4,Web Development,Web Design,Digital Marketing,Digital Marketing,Web Design,Web Design,Resume Review,Email Management,Search Engine Optimization (SEO)
5,Social Media,Graphic Design,Web Content Writing,Graphic Design,Cascading Style Sheets (CSS),Cascading Style Sheets (CSS),Curriculum Vitae (CV),Web Development,Responsive Web Design
6,Creative Writing,Social Media Marketing,Web Design,Web Content Writing,JavaScript,JavaScript,Resumes,Responsive Web Design,Front-End Development
7,Networking,Editing,Telecommunications,Web Design,Creative Writing,Search Engine Optimization (SEO),Screening Resumes,Web Design,Web Design
8,Social Media Marketing,Blogging,Editing,JavaScript,Copywriting,Web Development,Search Engine Optimization (SEO),Web Content Writing,Web Development
9,Blogging,Social Media,Writing,Creative Writing,Search Engine Optimization (SEO),Creative Writing,Applicant Tracking Systems,Front-End Development,Social Media Management


,2017,2018,2019,2020,2021,2022,2023,2024,2025
Rank,,,,,,,,,
1,Core Java,Core Java,Core Java,Core Java,Core Java,Core Java,Core Java,Core Java,Amazon Web Services (AWS)
2,SQL,SQL,Python (Programming Language),Python (Programming Language),Java,Java,Java,Amazon Web Services (AWS),Core Java
3,C (Programming Language),Java,C (Programming Language),C (Programming Language),SQL,SQL,REST APIs,Java,Java
4,Java,C (Programming Language),SQL,Data Structures,Python (Programming Language),Amazon Web Services (AWS),Amazon Web Services (AWS),REST APIs,REST APIs
5,Software Development Life Cycle (SDLC),JavaScript,Java,Java,Spring Boot,Spring Boot,SQL,SQL,SQL
6,JavaScript,Python (Programming Language),Data Structures,Amazon Web Services (AWS),C (Programming Language),Python (Programming Language),Spring Boot,React.js,React.js
7,HTML,HTML,JavaScript,SQL,Amazon Web Services (AWS),JavaScript,JavaScript,Spring Boot,Spring Boot
8,Requirements Analysis,Software Development Life Cycle (SDLC),Amazon Web Services (AWS),Spring Boot,JavaScript,REST APIs,Cascading Style Sheets (CSS),JavaScript,Python (Programming Language)
9,C++,Agile Methodologies,Spring Boot,JavaScript,MySQL,Cascading Style Sheets (CSS),Git,Git,Git


,2017,2018,2019,2020,2021,2022,2023,2024,2025
Rank,,,,,,,,,
1,Editing,Editing,Editing,Client Rapport,Agile Application Development,Editing,Search Engine Optimization (SEO),Swahili,Swahili
2,Video Production,Video Production,Python (Programming Language),Exceeding Customer Expectations,Agile Project Management,Virtual Assistance,React.js,Virtual Assistance,Virtual Assistance
3,Broadcasting,Video Editing,Video Production,Personal Development,Kanban,Data Entry,Swahili,Search Engine Optimization (SEO),Data Annotation
4,Video Editing,Social Media Marketing,Telecommunications,Customer Communication,Lean Management,Python (Programming Language),JavaScript,Virtual Administrative Support,Virtual Administrative Support
5,Journalism,Networking,Web Development,Career Management,DevOps,JavaScript,Editing,Python (Programming Language),Search Engine Optimization (SEO)
6,Creative Writing,Creative Writing,Journalism,Design Thinking,Scrum,Web Content Writing,Python (Programming Language),Typing,Web Development
7,Television,Journalism,Video Editing,Active Listening,Agile Methodologies,React.js,Software Development,Data Annotation,React.js
8,Research Skills,Telecommunications,JavaScript,Life Skills,Customer Focused Design,SEO Copywriting,Web Development,React.js,Full-Stack Development
9,Telecommunications,Research Skills,Cloud Computing,Leading Positive Change,Design Thinking,Transcription,Web Content Writing,Data Entry,Software Development


,2017,2018,2019,2020,2021,2022,2023,2024
Rank,,,,,,,,
1,Telecommunications,Telecommunications,Telecommunications,Telecommunications,Afrikaans,Telecommunications,Afrikaans,Afrikaans
2,Wireless Technologies,Wireless Technologies,Afrikaans,Afrikaans,Telecommunications,Computer Literacy,Attention to Detail,Attention to Detail
3,Voice over IP (VoIP),Troubleshooting,Technical Support,Editing,Computer Literacy,Editing,Customer Support,Typing
4,Business Analysis,Business Analysis,Broadcasting,Broadcasting,Consumer Services,Customer Support,Computer Literacy,Service-Level Agreements (SLA)
5,Broadcasting,Social Media Marketing,Troubleshooting,Computer Literacy,Editing,Troubleshooting,Telecommunications,Microsoft Azure
6,Editing,Editing,Computer Literacy,Technical Support,Technical Support,Video Editing,Microsoft Azure,Computer Literacy
7,IT Integration,Software Development Life Cycle (SDLC),SQL,Writing,Video Editing,Amazon Web Services (AWS),Customer Experience,Phone Etiquette
8,Internet Protocol (IP),Broadcasting,Editing,Amazon Web Services (AWS),Electronic Data Capture (EDC),Technical Support,SQL,Customer Engagement
9,SQL,Networking,Wireless Technologies,Customer Experience,Information Technology,SQL,Service-Level Agreements (SLA),Customer Support


,2017,2018,2019,2020,2021,2022,2023,2024,2025
Rank,,,,,,,,,
1,Research Skills,Graphic Design,Graphic Design,Graphic Design,Graphic Design,Graphic Design,Advertising,Virtual Assistance,Graphic Design
2,Public Speaking,Research Skills,Research Skills,Ghana,Web Development,Online Advertising,Graphic Design,Data Entry,Research Skills
3,Graphic Design,Public Speaking,Data Entry,Research Skills,Information Technology,Advertising,Python (Programming Language),Graphic Design,Advertising
4,Teamwork,Social Media Marketing,Web Development,Information Technology,Research Skills,Data Entry,Data Entry,Advertising,Ghana
5,Data Analysis,Teamwork,Data Collection,Web Development,Consumer Services,Adobe Photoshop,Online Advertising,Research Skills,Virtual Assistance
6,Web Development,Web Development,Public Speaking,Digital Marketing,Advertising,Python (Programming Language),Research Skills,Google Workspace,Data Entry
7,Social Media Marketing,Data Analysis,Web Design,Advertising,Web Design,Web Development,Web Development,Python (Programming Language),Web Development
8,Advertising,Social Media,Financial Reporting,Data Collection,Digital Marketing,Research Skills,Adobe Photoshop,Artificial Intelligence (AI),Python (Programming Language)
9,Financial Reporting,Financial Reporting,Data Analysis,Data Entry,Adobe Photoshop,Data Analysis,Typing,Typing,Records Management


,2017,2018,2019,2020,2021,2022,2023,2024,2025
Rank,,,,,,,,,
1,Legal Research,Legal Research,Graphic Design,Graphic Design,Graphic Design,Graphic Design,Resume Writing,Virtual Assistance,Virtual Assistance
2,Legal Writing,Legal Writing,Legal Research,Digital Marketing,Web Development,Online Advertising,Cover Letters,Virtual Administrative Support,Email Management
3,Litigation,Litigation,Digital Marketing,Web Development,Web Design,Virtual Assistance,Certified Professional Resume Writer,Email Management,Virtual Administrative Support
4,Research Skills,Graphic Design,Legal Writing,Web Design,Digital Marketing,Copywriting,Resume Review,Search Engine Optimization (SEO),Graphic Design
5,Legal Advice,Legal Advice,Web Development,Legal Research,User Interface Design,Web Design,Search Engine Optimization (SEO),Google Workspace,Search Engine Optimization (SEO)
6,Corporate Law,Web Development,Web Design,Writing,Cascading Style Sheets (CSS),Web Content Writing,Graphic Design,Graphic Design,Executive Calendar Management
7,Civil Litigation,Corporate Law,Litigation,JavaScript,JavaScript,User Interface Design,Online Advertising,Executive Calendar Management,Social Media Management
8,Graphic Design,Research Skills,Research Skills,Legal Writing,Copywriting,Cascading Style Sheets (CSS),Advertising,Data Entry,Google Workspace
9,Web Development,Web Design,Social Media Marketing,Research Skills,Graphics,Search Engine Optimization (SEO),Copywriting,Social Media Management,Data Entry


,2017,2018,2019,2020,2021,2022,2023,2024,2025
Rank,,,,,,,,,
1,SQL,Core Java,Core Java,Core Java,Core Java,Core Java,Core Java,Core Java,Core Java
2,Core Java,SQL,SQL,SQL,SQL,SQL,SQL,SQL,Amazon Web Services (AWS)
3,Software Development Life Cycle (SDLC),Java,Java,Java,Java,Java,Amazon Web Services (AWS),Amazon Web Services (AWS),SQL
4,Requirements Analysis,JavaScript,JavaScript,Python (Programming Language),Microsoft Azure,Manual Testing,Java,Java,Hindi
5,JavaScript,HTML,C (Programming Language),C (Programming Language),Manual Testing,Amazon Web Services (AWS),REST APIs,REST APIs,REST APIs
6,HTML,Requirements Analysis,Python (Programming Language),JavaScript,Amazon Web Services (AWS),Spring Boot,Cascading Style Sheets (CSS),JavaScript,Java
7,Java,Software Development Life Cycle (SDLC),HTML,Amazon Web Services (AWS),Spring Boot,JavaScript,Manual Testing,Cascading Style Sheets (CSS),Tally ERP
8,C (Programming Language),C (Programming Language),Amazon Web Services (AWS),Spring Boot,JavaScript,Jira,Jira,Git,Git
9,Team Management,Agile Methodologies,Cascading Style Sheets (CSS),Cascading Style Sheets (CSS),Python (Programming Language),Selenium,JavaScript,React.js,Jenkins


,2017,2018,2019,2020,2021,2022,2023,2024,2025
Rank,,,,,,,,,
1,Research Skills,Research Skills,Data Entry,Data Entry,Certified Public Accounting,Data Entry,Data Entry,Swahili,Swahili
2,Financial Accounting,Data Entry,Research Skills,Report Writing,Data Entry,QuickBooks,Swahili,Virtual Assistance,Virtual Assistance
3,Windows,Data Analysis,Data Collection,Research Skills,Swahili,Data Collection,Search Engine Optimization (SEO),Data Entry,Executive Calendar Management
4,Financial Reporting,Financial Accounting,Report Writing,Data Collection,Information Technology,Legal Research,QuickBooks,Data Annotation,Data Annotation
5,Data Analysis,Legal Research,QuickBooks,QuickBooks,Web Development,Virtual Assistance,Data Collection,Typing,Virtual Administrative Support
6,Legal Research,Social Media Marketing,Legal Research,Information Technology,Research Skills,Conveyancing,Legal Research,Online Data Entry,Data Entry
7,Data Entry,Web Development,Swahili,Legal Research,QuickBooks,Report Writing,Virtual Assistance,Search Engine Optimization (SEO),Search Engine Optimization (SEO)
8,System Administration,Financial Reporting,Data Analysis,Data Analysis,Data Analysis,Search Engine Optimization (SEO),Certified Public Accounting,Executive Calendar Management,Records Management
9,Account Reconciliation,HTML,Web Development,Writing,Report Writing,Python (Programming Language),Python (Programming Language),Email Management,Stakeholder Engagement


,2017,2018,2019,2020,2021,2022,2023,2024
Rank,,,,,,,,
1,Pastel Accounting,Pastel Accounting,Pastel Accounting,Pastel Accounting,Pastel Accounting,Pastel Accounting,Pastel Accounting,Afrikaans
2,Financial Accounting,Financial Accounting,Afrikaans,Afrikaans,Afrikaans,Afrikaans,Afrikaans,Pastel Accounting
3,Business Analysis,Financial Reporting,CaseWare Software,CaseWare Software,CaseWare Software,CaseWare Software,Attention to Detail,Attention to Detail
4,Legal Research,Pastel Partner,Pastel Partner,Computer Literacy,Pastel Partner,Computer Literacy,Computer Literacy,Electronic Data Capture (EDC)
5,Financial Reporting,Social Media Marketing,Computer Literacy,Pastel Partner,Computer Literacy,Electronic Data Capture (EDC),Typing,Typing
6,International Financial Reporting Standards (IFRS),Tax,Financial Reporting,Litigation,Electronic Data Capture (EDC),Office Administration,Electronic Data Capture (EDC),Computer Literacy
7,Legal Writing,Civil Litigation,Electronic Data Capture (EDC),Auditing,Sage Payroll,Pastel Partner,CaseWare Software,Phone Etiquette
8,Pastel Partner,Legal Research,Legal Research,Legal Writing,Office Administration,Sage Payroll,Phone Etiquette,Xero
9,Auditing,Business Analysis,Litigation,International Financial Reporting Standards (IFRS),Legal Writing,Legal Writing,Legal Writing,Skilled Multi-tasker


## Cross-industry transferability of target-sector skills

The existing ranked tables show which skills recur across countries and years. These additional tables keep the same rank-by-country layout but shade each skill according to how widely it appears outside its target industry. No shading means no other industry, light blue means 1–3 other industries, and dark blue means 4 or more.

The number in parentheses is a pooled five-country count, not a count for the country shown at the top of that column. For each skill, the calculation combines Ghana, Nigeria, India, Kenya, and South Africa, then counts the distinct LinkedIn industries, excluding the target industry, in which that skill appears. The same pooled count is therefore repeated wherever the skill appears in the country columns. For example, `React.js (4)` means React.js appears in four other industries across the five countries combined; it does not mean that it appears in four other industries within Ghana.

Because Skills Genome contains ranked characteristic skills rather than every skill held by every member, this is evidence of cross-industry visibility, not a direct measure of workers' ability to transfer between jobs.

In [12]:
target_skill_origins = (
    pooled_skills.groupby(["Industry", "Skill"], as_index=False)
    .agg(
        **{
            "Target Countries": ("Country", "nunique"),
            "Best Target Rank": ("Skill Rank", "min"),
        }
    )
    .rename(columns={"Industry": "Target Industry"})
)

transferability_occurrences = (
    target_skill_origins[["Target Industry", "Skill"]]
    .merge(pooled_skill_genome, on="Skill", how="inner")
    .loc[lambda d: d["Industry"].ne(d["Target Industry"])]
)
transferability_locations = (
    transferability_occurrences.groupby(
        ["Target Industry", "Skill", "Industry"], as_index=False
    )
    .agg(
        **{
            "Countries Reporting": ("Country", "nunique"),
            "Best Other-Industry Rank": ("Skill Rank", "min"),
        }
    )
    .rename(columns={"Industry": "Other Industry"})
)

transferability_summary = (
    target_skill_origins.merge(
        transferability_occurrences.groupby(["Target Industry", "Skill"])
        .agg(
            **{
                "Other Industries": ("Industry", "nunique"),
                "Other Countries": ("Country", "nunique"),
            }
        )
        .reset_index(),
        on=["Target Industry", "Skill"],
        how="left",
    )
    .fillna({"Other Industries": 0, "Other Countries": 0})
    .assign(
        **{
            "Other Industries": lambda d: d["Other Industries"].astype(int),
            "Other Countries": lambda d: d["Other Countries"].astype(int),
        }
    )
)
transferability_locations = transferability_locations.merge(
    transferability_summary[
        ["Target Industry", "Skill", "Other Industries", "Target Countries"]
    ],
    on=["Target Industry", "Skill"],
    how="left",
)

transferability_summary.to_csv(
    PROCESSED_PATH / "target_skill_transferability_summary.csv", index=False
)
transferability_locations.to_csv(
    PROCESSED_PATH / "target_skill_transferability_locations.csv", index=False
)


def transferability_skill_table(target_industry):
    """Show pooled ranks with shading based on cross-industry breadth."""
    summary = transferability_summary.loc[
        transferability_summary["Target Industry"].eq(target_industry)
    ]
    counts = summary.set_index("Skill")["Other Industries"].to_dict()
    industry_skills = pooled_skills.loc[pooled_skills["Industry"].eq(target_industry)]
    countries = [c for c in COUNTRIES if c in set(industry_skills["Country"])]
    grid = (
        industry_skills.assign(
            Country=lambda d: pd.Categorical(
                d["Country"], categories=countries, ordered=True
            )
        )
        .sort_values(["Country", "Skill Rank"])
        .pivot(index="Skill Rank", columns="Country", values="Skill")
        .rename_axis(index="Rank", columns=None)
    )

    def label(skill):
        return skill if pd.isna(skill) else f"{skill} ({counts.get(skill, 0)})"

    def shade(labelled_skill):
        if pd.isna(labelled_skill):
            return ""
        count = int(labelled_skill.rsplit("(", 1)[1].rstrip(")"))
        if count == 0:
            return "background-color: white; color: #374151"
        if count <= 3:
            return "background-color: #dbeafe; color: #1e3a5f"
        return "background-color: #2563eb; color: white"

    return (
        grid.map(label)
        .style.map(shade)
        .set_caption(f"Cross-industry transferability: {target_industry}")
        .set_properties(
            **{
                "font-size": "11px",
                "padding": "4px 6px",
                "border": "1px solid white",
            }
        )
        .set_table_styles(
            [
                {"selector": "th", "props": [("font-size", "11px")]},
                {"selector": "caption", "props": [("caption-side", "top")]},
            ]
        )
    )


for industry in TARGET_INDUSTRIES:
    display(transferability_skill_table(industry))

,Ghana,Nigeria,India,Kenya,South Africa
Rank,,,,,
1,React.js (4),Virtual Assistance (14),Core Java (8),Swahili (18),Afrikaans (19)
2,Front-End Development (1),React.js (4),Amazon Web Services (AWS) (2),Virtual Assistance (14),Telecommunications (0)
3,Node.js (1),Virtual Administrative Support (8),Spring Boot (2),React.js (4),Broadcasting (0)
4,Web Development (1),Search Engine Optimization (SEO) (3),Java (5),Search Engine Optimization (SEO) (3),Wireless Technologies (0)
5,Broadcasting (0),Front-End Development (1),SQL (5),Python (Programming Language) (5),Voice over IP (VoIP) (0)
6,JavaScript (4),Web Content Writing (4),Jenkins (1),Agile Application Development (0),Software Development Life Cycle (SDLC) (3)
7,Telecommunications (0),Responsive Web Design (1),REST APIs (1),Editing (1),IT Integration (1)
8,Python (Programming Language) (5),Creative Writing (6),C (Programming Language) (7),Virtual Administrative Support (8),Television (1)
9,Cascading Style Sheets (CSS) (4),User Interface Design (1),Data Structures (5),Exceeding Customer Expectations (0),Managed Services (0)


,Ghana,Nigeria,India,Kenya,South Africa
Rank,,,,,
1,Graphic Design (2),Virtual Assistance (14),Core Java (8),Swahili (18),Pastel Accounting (15)
2,Ghana (9),Virtual Administrative Support (8),SQL (5),QuickBooks (5),Afrikaans (19)
3,Python (Programming Language) (5),Search Engine Optimization (SEO) (3),Amazon Web Services (AWS) (2),Virtual Assistance (14),Pastel Partner (6)
4,Web Development (1),Legal Research (0),Tally ERP (18),Conveyancing (0),CaseWare Software (0)
5,Data Entry (8),Graphic Design (2),Java (5),Data Annotation (1),Electronic Data Capture (EDC) (11)
6,Tally ERP (18),User Interface Design (1),Jenkins (1),Legal Research (0),Civil Litigation (0)
7,Advertising (5),Email Management (7),Spring Boot (2),Data Entry (8),Litigation (0)
8,JavaScript (4),Legal Writing (0),Manual Testing (1),Search Engine Optimization (SEO) (3),Legal Research (0)
9,Front-End Development (1),User Experience Design (UED) (1),Jira (1),Certified Public Accounting (3),Legal Writing (0)
